# Use Case 1 - Name/M3 --> Proposal/Teams

<ol>
    <li>Generic Data Import. <a href="#gen_data_import">Here.</a></li>
    <li>Method/Files Import. <a href="#method_import">Here.</a></li>
    <li>M3 - Boosted Bandit Matching. <a href="#m0">Here.</a></li>
</ol>

## Generic Data Import <a id='gen_data_import'></a> 

In [1]:
import pandas as pd
import numpy as np
# Import list of researchers
og_researchers=pd.read_csv('../data/v1_input_files/v1_researchers.csv')

# Import a subset of the proposals, sorted by the YEAR they were sent
proposal_info=pd.read_csv('../data/v1_input_files/v1_proposal_links_title_synopsis.csv')
proposal_info.sort_values(["nsf_proposal_links_v1"], 
                    ascending=[False], 
                    inplace=True)
# proposal_info=proposal_info[:100]
proposal_info.pop("Unnamed: 0")
proposal_info.reset_index(drop=True, inplace=True)

In [2]:
linebreak = "\n\n----------------------------------------------------------------"

print(og_researchers.iloc[0], linebreak)
print(proposal_info.iloc[0])

Unnamed: 0                                                      0
names                                         Agostinelli, Forest
descriptions                                  Assistant Professor
titles                                                    Faculty
research        ['Artificial Intelligence, Deep Learning, Rein...
Name: 0, dtype: object 

----------------------------------------------------------------
nsf_proposal_links_v1    https://www.nsf.gov/pubs/2021/nsf21598/nsf2159...
title                                     Advanced Technological Education
synopsis                 With a focus on two-year Institutions of Highe...
Name: 0, dtype: object


In [3]:
# error case encountered at the end - so preventing it now by excluding this proposal

error_values = ['https://www.nsf.gov/funding/pgm_summ.jsp?pims_id=505073'] 

#drop rows that contain any value in the list
proposal_info = proposal_info[proposal_info.nsf_proposal_links_v1.isin(error_values) == False]
proposal_info.reset_index(drop=True, inplace=True)

***
## Methods/Files Import <a id='method_import'></a> 

In [4]:
import nlp_techniques
import M3
import importlib
importlib.reload(M3)

[nltk_data] Downloading package wordnet to /Users/tej/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/tej/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


<module 'M3' from '/Users/tej/Desktop/Recommendation_Projects/Teaming/code/M3.py'>

## M3 - Boosted Bandit Matching <a id="m0"></a>

<b>Steps:</b>
<ol>
    <li>Preprocess the generated predicates. <a href="#method_import_1">Here.</a></li>
    <li>Train/test the data by running BoostSRL.jar on both of the directories. <a href="#method_import_2">Here.</a></li>
    <li>Open the generated <code>results_team.db</code> file and extracting teaming info. <a href="#method_import_3">Here.</a></li>
    <li>Form teams based on the generated information. <a href="#method_import_4">Here.</a></li>
    <li>Export data to CSV. <a href="#method_import_5">Here.</a></li>    
</ol>

### Step 1 - Preprocess the generated predicates. <a id='method_import_1'></a> 

First, run `prepare.py` in "./boosted_results/". This will generate facts, positive pairs, and negative pairs. A fact is defined as what we already know. We know a researcher's personal skills as well as a proposal's demanded requirements. A positive/negative pair is defined as an association (or the lack of it) between a certain researchere and a proposal. 

Example of each:
<ul>
    <li>Fact: <code>skill(nsf18585,division).</code></li>
    <li>Pos: <code>team(nsf18595,reynolds_tony).</code></li>
    <li>Neg: <code>team(nsf20540,lyons_jed).</code></li>
</ul>

Due to the nature of proposals/researchers CSV files, some predicates have misplaced/extra characters. And the boostedsrl.jar file cannot parse them properly, so this step fixes that issue.

In [5]:
import importlib
importlib.reload(M3)

# directories where the boosted code/data is generated and run
homedir="./boosted_results/"
traindir=homedir+"train/"
testdir=homedir+"test/"

# for each of the facts/pos/neg files, clean them such the boosted bandit model (boostedsrl.jar) can parse them
import os

iteration_needed=[traindir, testdir]

for i in iteration_needed:   # iterate over both train and testdirs
    for filename in os.listdir(i):      # make sure to only check facts/pos/neg files
        if filename.endswith('facts.txt') or filename.endswith('pos.txt') or filename.endswith('neg.txt'):
            
            destination_file=i+filename
                
            # clean each line    
            with open(i+filename,'r') as file:     # open file
                lines=[line.rstrip() for line in file]      # get all lines of text
                for j in range(len(lines)):   
                    lines[j]=M3.predicate_clean(lines[j])
                
            # write lines back into the destination file
            destination_file=i+filename
            with open(destination_file, 'w') as file:
                for line in lines:
                    file.write(f"{line}\n")

## Step 2 - Train/test the data by running BoostSRL.jar on both of the directories. <a id='method_import_2'></a> 

Run the `boostedsrl.jar` file for each of the training and testing directories.
<ul>
    <li><code>java -jar 'boostsrl_v1.1.1.jar' -l -combine -train train/ -target team -trees 20</code></li>
    <li><code>java -jar 'boostsrl_v1.1.1.jar' -i -model train/models/ -test test/ -target team -aucJarPath . -trees 20</code></li>
        

### Step 3 - Open the generated <code>results_team.db</code> file and extracting teaming info. <a id='method_import_3'></a> 

In [6]:
# directories where the boosted code/data is generated and run
homedir="./boosted_results/"
traindir=homedir+"train/"
testdir=homedir+"test/"

# results file
results=testdir+"results_team.db"

pos_teams={}    # an association between a researcher and a proposal. E.g., "team(nsf12345,researcher)"
neg_teams={}    # a rejection between a researcher and a proposal. E.g., "!team(nsf12345,researcher)"

with open(results,'r') as file:     # open file
    lines=[line.rstrip() for line in file]      # get all lines of text
    
    for line in lines:
        
        # check which of the pos/neg associations a pairing has
        if line.startswith("team"):    # positive association
            posFlag=True
        elif line.startswith("!team"):    # negative association
            posFlag=False
            
        # strip the line of whitespace and extras
        tokens=line.split(" ")
        
        if posFlag:
            tokens[0]=tokens[0][5:-1]    # remove the 'team(' or '!team(' part
        else:
            tokens[0]=tokens[0][6:-1]
        
        tokens[1]=tokens[1][:-1]   # remove the ending parentheses ')'
        tokens[2]=float(tokens[2])
        
        # save tokens
        proposal=tokens[0]
        researcher=tokens[1]
        likelihood=tokens[2]
        
        # add the tokens to pos_teams{} or neg_teams{}, depending on flag
        if posFlag:
            if proposal not in pos_teams.keys():
                pos_teams[proposal]={}
                
            pos_teams[proposal][researcher]=likelihood
        else:
            if proposal not in neg_teams.keys():
                neg_teams[proposal]={}
            
            neg_teams[proposal][researcher]=likelihood

### Step 4 - Extract and preprocess a researcher's own skills (extracted from their homepage in the faculty directory). <a id='method_import_4'></a> 

In [7]:
import ast
import datetime

# Start time
print("Start time:\t", datetime.datetime.now())

m3_researcher_skills={}

# for each researcher
for i in range(len(og_researchers["research"])):
    # load into variables
    researcher=og_researchers["names"][i]
    interests=og_researchers["research"][i]
    if type(interests)==float:  # account for nan values
        interests="['research', 'general', 'computer', 'science', 'engineering']"
    
    # convert the string of interests into a list of interests
    interests=ast.literal_eval(interests)[0].split(", ")
    for j in range(len(interests)):
        interests[j]=nlp_techniques.preprocess(interests[j])
        
    while '' in interests:
        interests.remove('')
        
    # print(researcher,interests)   # E.g., Agostinelli, Forest ['artificial intelligence', 'deep learning', ...]

    # do it for n-grams of 2 as well
    n_gram_interests=[]
    for j in range(len(interests)):
        n_gram_interests.append(nlp_techniques.generate_N_grams(interests[j], ngram=2))
        
    n_gram_interests=[item for sublist in n_gram_interests for item in sublist]   # merge list of lists into a flat list
    
    # merge interests
    merged_interests=set(interests+n_gram_interests)
    while '' in merged_interests:
        merged_interests.remove('')    # remove null/empty strings
    
    # save info
    m3_researcher_skills[researcher]=merged_interests
    
# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:28:32.978054
End time:	 2026-04-01 15:28:36.653233


In [8]:
# save directory
save_dir="../data/v1_output_teaming/teaming_1698proposals_316researchers//data_uc1_m3/"

# export m3_researcher_skills
csv_m3_researcher_skills=[]
for i in m3_researcher_skills:
    csv_m3_researcher_skills.append([i, m3_researcher_skills[i]])

csv_m3_researcher_skills=pd.DataFrame(csv_m3_researcher_skills, columns = ['researcher_name', 'skills'])
csv_m3_researcher_skills.to_csv(save_dir+'m3_researcher_skills.csv', encoding='utf-8')
del csv_m3_researcher_skills

### Step 5 - Store all researchers' skills in a separate list. <a id='method_import_5'></a> 

**Why?**

Because a proposal individually would have a lot of "skills" extracted, including the irrelevant terms (or those that the researchers are **not** familiar with). If a proposal has N skills extracted from its title/synopsis, then each of the i-th skill would only count IF that i-th skill is already in m3_all_researcher_skills[] (below).

This way, we are condensing the number of searches, and making Ultra-Metric more readable.

In [9]:
import datetime 

# Start time
print("Start time:\t", datetime.datetime.now())

# set of all skills that researchers have
m3_all_researcher_skills=[]

for i in m3_researcher_skills:    # compile a list of all skills that researchers have
    for j in m3_researcher_skills[i]:
        if j not in m3_all_researcher_skills:
            m3_all_researcher_skills.append(j)

# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:28:36.673172
End time:	 2026-04-01 15:28:37.007339


In [10]:
m3_all_researcher_skills

['reinforcement learning',
 'artificial intelligence',
 'deep learning',
 'search',
 'bioinformatics',
 'electronic technology',
 'tech universityresearch',
 'inc ph',
 'fabrication novel high power electronic photonic device',
 'universityresearch growth',
 'previous position',
 'content algan',
 'novel high',
 'computer simulation',
 'nitride gallium',
 'ph texas',
 'senior scientist',
 'bandgap semiconductor',
 'growth study',
 'study ultra',
 'high aluminum',
 'gallium oxide',
 'fabrication novel',
 'boron nitride',
 'semiconductor including',
 'computer simulation device',
 'previous position senior scientist',
 'simulation device',
 'boron nitride gallium oxide',
 'including high',
 'wide bandgap',
 'electronic photonic',
 'inc ph texas tech universityresearch growth study ultra wide bandgap semiconductor including high aluminum content algan',
 'sensor electronic',
 'photonic device',
 'ultra wide',
 'power electronic',
 'high power',
 'texas tech',
 'aluminum content',
 'positi

In [11]:
# save directory
save_dir="../data/v1_output_teaming/teaming_1698proposals_316researchers/data_uc1_m3/"

# export m3_all_researcher_skills
csv_m3_all_researcher_skills=[]
for i in m3_all_researcher_skills:
    csv_m3_all_researcher_skills.append(i)

csv_m3_proposal_skills=pd.DataFrame(csv_m3_all_researcher_skills, columns = ['all_skills'])
csv_m3_proposal_skills.to_csv(save_dir+'m3_all_researcher_skills.csv', encoding='utf-8')
del csv_m3_all_researcher_skills

### Step 6 - Extract necessary "skills" required for each proposal, with everyone as a whole having them. <a id='method_import_6'></a> 

In [12]:
# Start time
print("Start time:\t", datetime.datetime.now())

m3_proposal_skills={}

# for each proposal
for i in range(len(proposal_info["nsf_proposal_links_v1"])):
    # extract respective title/synopsis
    title=proposal_info["title"][i]
    synopsis=proposal_info["synopsis"][i]
        
    # check if title is an empty field
    if type(title)==float:     # if so, then assign a general value to title
        title="general"
    
    # check if synopsis is an empty field
    if type(synopsis)==float:     # if so, then assign an empty value to synopsis
        synopsis=""
    
    # preprocess title/synopsis
    title=nlp_techniques.preprocess(title)
    synopsis=nlp_techniques.preprocess(synopsis)
        
    # keywords of title - just split the string and apply set()
    keywords=title.split(" ")+synopsis.split(" ")

    # n-gram keywords
    title_n=nlp_techniques.generate_N_grams(title, ngram=2)
    synopsis_n=nlp_techniques.generate_N_grams(synopsis, ngram=2)
    
    n_gram_keywords=set(title+synopsis)
    
    # merge
    all_keywords=set(keywords+title_n+synopsis_n)
    
    # if any of these keywords do not exist in m3_all_researcher_skills, remove them
    skills_to_be_removed=[]
    for j in all_keywords:
        if j not in m3_all_researcher_skills:
            skills_to_be_removed.append(j)
            
    for j in skills_to_be_removed:
        all_keywords.remove(j)        
    
    # in case of empty set
    if all_keywords==set():
        all_keywords=set(["general"])
        
    # add them to the dictionary mapping
    m3_proposal_skills[proposal_info["nsf_proposal_links_v1"][i]]=all_keywords
    
# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:28:37.050380
End time:	 2026-04-01 15:28:44.435173


In [13]:
# export m3_proposal_skills
csv_m3_proposal_skills=[]
for i in m3_proposal_skills:
    csv_m3_proposal_skills.append([i, m3_proposal_skills[i]])

csv_m3_proposal_skills=pd.DataFrame(csv_m3_proposal_skills, columns = ['nsf_proposal_links_v1', 'skills'])
csv_m3_proposal_skills.to_csv(save_dir+'m3_proposal_skills.csv', encoding='utf-8')
del csv_m3_proposal_skills

### Step 7 - Map researcher names to parser-understandable denotions <a id='method_import_7'></a> 

E.g., Parser converts "Agostinelli, Forest" to "agostinelli_forest"

In [14]:
researchers={}

list_of_researchers=m3_researcher_skills.keys()

for i in list_of_researchers:    # name
    
    # convert 
    name = i.strip().replace(',', '')\
        .replace('.', '')\
        .replace('(', '')\
        .replace(')', '')\
        .replace(' ', '_')\
        .replace('-', '_')\
        .replace('\'', '_')\
        .lower()
    
    # save
    researchers[name]=i

### Step 8 - Form teams based on the generated information. <a id='method_import_8'></a> 

In [15]:
string_matching_threshold=0.3

In [16]:
import random
import importlib
importlib.reload(M3)

import datetime

# Start time
print("Start time:\t", datetime.datetime.now())

# Code
m3_teaming={}
m3_pseudo_researcher_skills={}

# for each proposal 
for i in range(len(proposal_info["nsf_proposal_links_v1"])):
    
    # get proposal_id from index
    proposal=proposal_info["nsf_proposal_links_v1"][i]
    proposal_id=proposal.split("/")[5]
    
    # initialize m3_teaming
    m3_teaming[proposal]=[]
    
    # because we're using a threshold count, skills that a researcher never had may also get selected. We keep track of this in a separate list
    m3_pseudo_researcher_skills[proposal]={}    
    count=0
    
    # for each researcher, create N teams`based on boosted bandit
    for j in range(0,len(og_researchers["names"])):
        target_researcher=og_researchers["names"][j]
        
        # call m3 function - string_matching_ranking()
        if m3_pseudo_researcher_skills[proposal]=={}:
            ranking, pseudo_researcher_skills=M3.string_matching_ranking(m3_researcher_skills, m3_proposal_skills[proposal], {}, matching_threshold=string_matching_threshold)            
            m3_pseudo_researcher_skills[proposal]=pseudo_researcher_skills
        else: 
            ranking=M3.string_matching_ranking(m3_researcher_skills, m3_proposal_skills[proposal], pseudo_researcher_skills, matching_threshold=string_matching_threshold)
        
        # call M3 function - create_teams_for_each_person()
        num_of_teams=10
        teams=M3.create_teams_for_each_person(ranking, target_researcher, num_of_teams)
        
        # save teams in the researcher's profile
        m3_teaming[proposal].append([target_researcher,teams])
        
        # for each team, now check boosted bandit results
        if proposal_id not in pos_teams.keys() and proposal_id not in neg_teams.keys():
            continue
        
        bandit_teams=[]
        count=0
        
        try:
            pos_members=list(pos_teams[proposal_id].keys()) # select team members from bandit matches
        except:
            continue
        
        while count<num_of_teams:
            count+=1
            team=random.sample(pos_members, min(len(pos_members), 4))

            if target_researcher in team:# filter out from accidentally selecting the same team member twice within a team
                team.remove(target_researcher)

            team.insert(0, target_researcher) # add the target_researcher as part of the team as well
            
            if team not in bandit_teams:
                bandit_teams.append(team)
        
        if len(bandit_teams)<num_of_teams:
            difference=num_of_teams-len(bandit_teams)
            remaining_teams=teams[:difference]
            
            for j in range(len(remaining_teams)):
                for k in range(1, len(remaining_teams[j])):   # omit target researcher from checking in neg_teams
                    try:
                        watch_member=list(researchers.keys())[list(researchers.values()).index(remaining_teams[j][k])]
                    except:
                        continue
                    try:
                        if watch_member in list(neg_teams[proposal_id].keys()):
                            del remaining_teams[j][k]
                    except:
                        continue      
                        
        for j in range(len(bandit_teams)):
            for k in range(len(bandit_teams[j])):
                try:
                    bandit_teams[j][k]=researchers[bandit_teams[j][k]]
                except:
                    pass
        # print(bandit_teams)
        
        if proposal_id in pos_teams.keys():
            for team in teams:
                temp_team=[]
                for member in team:
                    temp_team.append(list(researchers.keys())[list(researchers.values()).index(member)])
                print(temp_team)
                break
        
        # save teams in the researcher's profile
        m3_teaming[proposal][-1][1]=bandit_teams
        
# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:28:44.471754
['agostinelli_forest', 'wang_xiaofeng', 'zhou_caizhi']
['ahmad_iftikhar', 'wang_guoan', 'yoon_yeomin', 'won_sang_hee']
['alexeev_oleg_s', 'xue_xingjian_chris', 'wang_song', 'yuan_lang']
['ali_mohammod', 'xue_xingjian_chris', 'white_ralph_e', 'zhou_caizhi']
['ammal_salai_c', 'xu_songhua', 'yuan_lang', 'yoon_yeomin', 'zand_ramtin']
['bakos_jason_d', 'zhou_caizhi', 'wang_song']
['banerjee_sourav', 'zhang_qi', 'zhou_caizhi', 'williams_christopher']
['bayat_mahmoud', 'xu_songhua', 'ziehl_paul', 'wells_james_r']
['bayoumi_abdel_moez_e', 'wang_song', 'yu_lingyu', 'zhou_caizhi', 'xue_xingjian_chris']
['benigni_andrea', 'zhang_bin', 'wang_song', 'zhang_qi']
['berge_nicole_d', 'zhang_qi', 'xu_songhua', 'wu_dezhi', 'wang_guoan']
['besmann_theodore__m', 'yuan_lang', 'wang_xiaofeng', 'williams_christopher']
['bischoff_jeff', 'wang_guoan']
['blanchette_james__otto', 'ziehl_paul', 'yu_lingyu']
['boltin_nicholas_d', 'ziehl_paul', 'wei_xiaojun']
['booth_kristen',

In [17]:
# export m3_teaming
csv_m3_teaming=[]
for i in m3_teaming:   # proposal
    for j in m3_teaming[i]:  # researcher
        for k in j[1]:         # list of teams
            csv_m3_teaming.append([i, j[0], k])

csv_m3_teaming=pd.DataFrame(csv_m3_teaming, columns = ['nsf_proposal_links_v1', 'researcher', 'team'])
csv_m3_teaming.to_csv(save_dir+'m3_teaming.csv', encoding='utf-8')
del csv_m3_teaming

### Step 9 - Apply Ultra-Metric. <a id='method_import_5'></a> 

In [18]:
import importlib
importlib.reload(M3)

# Start time
print("Start time:\t", datetime.datetime.now())

# import Ultra-Metric 
import metrics_scorer as metrics

m3_goodness_scores={}

# for each proposal
for proposal in m3_teaming:
    #initialize
    m3_goodness_scores[proposal]=[]
    
    # for each researcher
    for researcher in range(len(m3_teaming[proposal])):
        try:
            goodness_for_each_researcher=[m3_teaming[proposal][researcher][0],[]]

            # for each team
            for index in range(len(m3_teaming[proposal][researcher][1])):     # [0] - contains researcher's name, where [1] contains teams
                # initialize to team
                team=m3_teaming[proposal][researcher][1][index]

                # Apply Ultra-Metric (demand, team, researchers)
                temp_team_goodness=M3.apply_ultra_metric(m3_proposal_skills[proposal], list(set(team)), m3_pseudo_researcher_skills[proposal])

                # save to scores
                goodness_for_each_researcher[1].append(temp_team_goodness)

            # save to overall dictionary
            m3_goodness_scores[proposal].append(goodness_for_each_researcher)
        except:
            continue
    
# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:29:59.512656
End time:	 2026-04-01 15:31:35.545318


In [19]:
m3_proposal_skills[proposal], team

({'brown jr',
  'civil',
  'directorate engineering',
  'earthquake engineering',
  'engineering',
  'engineering simulation',
  'experimentation',
  'foundation nsf',
  'george brown',
  'geotechnical',
  'inc',
  'jr network',
  'mechanical system',
  'national science',
  'nationally internationally',
  'network earthquake',
  'nsf funding',
  'science foundation',
  'use experimental'},
 ['Ziehl, Paul', 'Humphries, Wilfred Kenneth', 'Gassman, Sarah'])

In [20]:
# export m3_goodness_scores
csv_m3_goodness_scores=[]
for i in m3_goodness_scores:
    for j in m3_goodness_scores[i]:
        for k in j[1]:
            csv_m3_goodness_scores.append([i, j[0], k])
            
csv_m3_goodness_scores=pd.DataFrame(csv_m3_goodness_scores, columns = ['nsf_proposal_links_v1', 'researcher_name', 'goodness'])
csv_m3_goodness_scores.to_csv(save_dir+'m3_goodness_scores.csv', encoding='utf-8')
del csv_m3_goodness_scores

### Step 10 - Final exports to CSV. <a id='method_import_10'></a> 

In [21]:
# data = [proposal_link, proposal_title, proposal_skills, researcher, team, goodness]

# Start time
print("Start time:\t", datetime.datetime.now())

# group together and export the teaming data
save_dir="../data/v1_output_teaming/teaming_1698proposals_316researchers/"
csv_uc1_m3_teaming=[]
for i in m3_teaming:    # proposal
    for j in range(len(m3_teaming[i])):     # researcher
        try:
            # get title
            title_index=list(proposal_info['nsf_proposal_links_v1']).index(i)
            title=proposal_info['title'][title_index]

            # formatting variables (proposal year, proposal ID, proposal name + year)
            year=i.split("/")[4]
            proposal_id=i.split("/")[5]      # nsf#####
            csv_hyperlink_text=proposal_info['title'][title_index]+" ("+str(year)+")"     # Sample Proposal Name (2023)

            # sort teams in descending order (based on goodness scores)
            unsorted_teams=m3_teaming[i][j][1]
            unsorted_goodness=m3_goodness_scores[i][j][1]

            sorted_teams=[x for _,x in sorted(zip(unsorted_goodness, unsorted_teams), reverse=True)]
            sorted_goodness=sorted(unsorted_goodness, reverse=True) 

            # round goodness scores
            rounded_scores=[]
            for score in sorted_goodness:
                rounded_scores.append(round(score,4))

            # save
            csv_uc1_m3_teaming.append([proposal_id,
                                       year,
                                       i,
                                       #"=HYPERLINK(\""+i+"\", \""+csv_hyperlink_text+"\")",
                                       csv_hyperlink_text,
                                       m3_proposal_skills[i], 
                                       m3_teaming[i][j][0], 
                                       sorted_teams, 
                                       rounded_scores])
        except:
            pass

csv_uc1_m3_teaming=pd.DataFrame(csv_uc1_m3_teaming, columns = ['proposal_id', 'year', 'proposal_link', 'title', "skills", "researcher_name", "team", "goodness"])
csv_uc1_m3_teaming.to_csv(save_dir+'teaming_uc1_m3.csv', encoding='utf-8')
#del csv_uc1_m3_teaming

# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:31:37.733125
End time:	 2026-04-01 15:31:45.461769


In [22]:
# --- M3 Skill Coverage Verification ---
# Replace with your specific proposal and team from the m3_teaming results
proposal_link = "https://www.nsf.gov/pubs/2013/nsf13543/nsf13543.htm"
team_members = ['Chen, Fanglin (Frank)']

# 1. Get the required skills for this proposal
required_skills = m3_proposal_skills[proposal_link]

# 2. Get the skills for each team member (using the M3 pseudo-skills mapping)
# Note: M3 uses pseudo-skills because of string matching thresholds
team_skill_pool = set()
for member in team_members:
    # Use pseudo_skills_map generated during the M3 run (Cell 15)
    member_skills = m3_pseudo_researcher_skills[proposal_link].get(member, set())
    team_skill_pool.update(member_skills)

# 3. Calculate Coverage
covered_skills = required_skills.intersection(team_skill_pool)
missing_skills = required_skills - team_skill_pool
coverage_percent = (len(covered_skills) / len(required_skills)) * 100 if required_skills else 0

# --- Print Results ---
print(f"Proposal: {proposal_link}")
print(f"Required Skills ({len(required_skills)}): {required_skills}")
print(f"Covered Skills ({len(covered_skills)}): {covered_skills}")
print(f"Missing Skills ({len(missing_skills)}): {missing_skills}")
print(f"Final Coverage: {coverage_percent:.2f}%")

Proposal: https://www.nsf.gov/pubs/2013/nsf13543/nsf13543.htm
Required Skills (15): {'national science', 'area', 'process modeling', 'development', 'clinical practice', 'machine learning', 'health', 'nih', 'science foundation', 'foundation nsf', 'decision support', 'modeling', 'support system', 'national institute', 'learning'}
Covered Skills (14): {'machine learning', 'national institute', 'development', 'modeling', 'process modeling', 'foundation nsf', 'support system', 'national science', 'decision support', 'learning', 'clinical practice', 'health', 'science foundation', 'area'}
Missing Skills (1): {'nih'}
Final Coverage: 93.33%


In [23]:
import pandas as pd
import numpy as np

# 1. Calculate 'Volume' (#T) per row
# 'team' is likely a list of teams. We count how many teams are in that list.
csv_uc1_m3_teaming['volume'] = csv_uc1_m3_teaming['team'].apply(lambda x: len(x))

# 2. Calculate 'Average Goodness' (G) per row
# 'goodness' is a list of scores. We take the mean of that list.
csv_uc1_m3_teaming['avg_goodness_per_row'] = csv_uc1_m3_teaming['goodness'].apply(lambda x: np.mean(x) if len(x) > 0 else 0)

# 3. Group by Researcher to get metrics per r_j
researcher_stats = csv_uc1_m3_teaming.groupby('researcher_name').agg({
    'avg_goodness_per_row': 'mean',
    'volume': 'mean'
}).reset_index()

# 4. Final Aggregation (The values for Table 3.6)
final_mean_g = researcher_stats['avg_goodness_per_row'].mean()
final_std_g = researcher_stats['avg_goodness_per_row'].std()
final_volume = researcher_stats['volume'].mean()

print(f"Average Goodness (G): {final_mean_g:.4f} ± {final_std_g:.4f}")
print(f"Average Volume (#T): {final_volume:.2f}")

Average Goodness (G): 0.5894 ± 0.0058
Average Volume (#T): 6.84


In [24]:
import pandas as pd
import numpy as np

# 1. Row-level metrics (assuming 'team' and 'goodness' are lists)
csv_uc1_m3_teaming['volume'] = csv_uc1_m3_teaming['team'].apply(len)
csv_uc1_m3_teaming['avg_goodness_per_row'] = csv_uc1_m3_teaming['goodness'].apply(lambda x: np.mean(x) if len(x) > 0 else 0)

# 2. Aggregation per Researcher
# We also count how many proposals each researcher was tested in
researcher_stats = csv_uc1_m3_teaming.groupby('researcher_name').agg({
    'avg_goodness_per_row': 'mean',
    'volume': 'mean',
    'proposal_id': 'count' # How many times this researcher was a 'seed'
}).rename(columns={
    'avg_goodness_per_row': 'mean_G', 
    'volume': 'mean_T',
    'proposal_id': 'proposal_count'
}).reset_index()

# 3. Print Global Table 3.6 Stats
final_mean_g = researcher_stats['mean_G'].mean()
final_std_g = researcher_stats['mean_G'].std()
final_volume = researcher_stats['mean_T'].mean()

print(f"--- GLOBAL TABLE 3.6 STATS ---")
print(f"Average Goodness (G): {final_mean_g:.4f} ± {final_std_g:.4f}")
print(f"Average Volume (#T): {final_volume:.2f}\n")

# 4. Detailed Researcher Info
print("--- TOP 5 'EASY TO TEAM' RESEARCHERS (Highest Goodness) ---")
print(researcher_stats.sort_values('mean_G', ascending=False).head(5))

print("\n--- BOTTOM 5 'HARD TO TEAM' RESEARCHERS (Lowest Goodness) ---")
print(researcher_stats.sort_values('mean_G', ascending=True).head(5))

# 5. Distribution of Volume
print("\n--- VOLUME DISTRIBUTION ---")
print(researcher_stats['mean_T'].describe())

--- GLOBAL TABLE 3.6 STATS ---
Average Goodness (G): 0.5894 ± 0.0058
Average Volume (#T): 6.84

--- TOP 5 'EASY TO TEAM' RESEARCHERS (Highest Goodness) ---
             researcher_name    mean_G    mean_T  proposal_count
13   Blanchette, James  Otto  0.595757  6.833333             432
119      Nezhad, Cyrus Riahi  0.595673  6.835648             432
129               Potts, Jay  0.595665  6.849537             432
86     Karakalos, Stavros G.  0.595632  6.858796             432
157        Spinale, Frank G.  0.595629  6.837963             432

--- BOTTOM 5 'HARD TO TEAM' RESEARCHERS (Lowest Goodness) ---
        researcher_name    mean_G    mean_T  proposal_count
120       O'Kane, Jason  0.567502  6.868056             432
55   Gibbons, Joseph H.  0.568440  6.847222             432
160      Stiffler, Nick  0.568899  6.856481             432
166         Tan, Wenbin  0.569330  6.863426             432
64       Gürdal, Zafer   0.573394  6.821759             432

--- VOLUME DISTRIBUTION ---
co